# Car price — Feature selection after dummy encoding (4 models)
Pipeline: raw CSV → drop `year` → split 80/10/10 → one-hot dummies → per-model selection. No other processing: no age filter, no model-name cleanup, no rare-level grouping.

- `year` never enters a model: `year = 2026 − car_age` exactly, so it carries no extra information.
- The split comes **before** dummies, so levels are learned from train only. Levels unseen in train become all-zero rows in val/test.
- Target = `log(price)` for every model (price is right-skewed). Metrics are reported on the price scale.
- Validation is used for every selection decision. Test is scored **once**, at the very end.

| Model | Selection | Decided on |
|---|---|---|
| MLR | VIF (iterative, < VIF_MAX) → Elastic Net (`l1_ratio` grid includes 1.0 = LASSO) | 5-fold CV on train |
| DT | Cost-complexity pruning; features not used by the pruned tree are dropped | validation |
| RF | Permutation importance; drop if mean − 2·SD ≤ 0 | validation |
| XGB | Early stopping + mean \|SHAP\| ranking; smallest top-k set within TOL of the best | validation |

In [1]:
import time, warnings, numpy as np, pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import config as C
from tree_data import score

pd.set_option("display.width", 200, "display.max_rows", 200)
warnings.filterwarnings("ignore")   # sklearn Parallel/delayed spam from RF.predict in the permutation loop
TOL = 0.02   # "near-best" = val MAE within 2% of the best (same rule as tree_data.pick)
raw = pd.read_csv(C.DATA_PATH).drop(columns="year")
train, tmp = train_test_split(raw, train_size=0.8, random_state=C.SEED)
val, test = train_test_split(tmp, test_size=0.5, random_state=C.SEED)
parts = dict(train=train, val=val, test=test)
y = {k: np.log(d[C.TARGET]) for k, d in parts.items()}
NUM = [c for c in raw.select_dtypes("number").columns if c != C.TARGET]
CAT = raw.select_dtypes(exclude="number").columns.tolist()
levels = {c: train[c].value_counts().index.tolist() for c in CAT}   # train only, most frequent first

def dummies(d, drop_first):
    X = d[NUM].astype(float)
    for c in CAT:   # unseen level -> NaN -> all-zero dummies
        D = pd.get_dummies(pd.Categorical(d[c].where(d[c].isin(levels[c])), categories=levels[c]), prefix=c, dtype=float).set_index(d.index)
        X = X.join(D.iloc[:, 1:] if drop_first else D)   # drop_first drops the most frequent level (reference)
    return X

orig = lambda c: c if c in NUM else next(v for v in CAT if c.startswith(v + "_"))
def kept_by_var(kept, allcols):
    k, a = pd.Series(map(orig, kept)).value_counts(), pd.Series(map(orig, allcols)).value_counts()
    return k.reindex(a.index, fill_value=0).astype(str) + "/" + a.astype(str)
vmae = lambda pred_log, part="val": mean_absolute_error(parts[part][C.TARGET], np.exp(pred_log))
summary = []

print({k: len(d) for k, d in parts.items()}, "| numeric:", NUM, "| categorical:", {c: len(levels[c]) for c in CAT})
print("rows with a level unseen in train:", {k: int(sum((~d[c].isin(levels[c])).sum() for c in CAT)) for k, d in parts.items() if k != "train"})

{'train': 24083, 'val': 3010, 'test': 3011} | numeric: ['engine_capacity', 'mileage', 'car_age'] | categorical: {'brand': 52, 'model': 520, 'car_type': 7, 'fuel_type': 3, 'gear_type': 2, 'color': 21}
rows with a level unseen in train: {'val': 15, 'test': 20}


## 1. MLR — VIF, then Elastic Net
**Step 1: VIF.** OLS needs a full-rank, low-collinearity X, or coefficients become unstable and uninterpretable. Dummies create **exact** dependencies: a brand dummy equals the sum of its models' dummies, and `car_type` is nested in `model`.
- Exact dependency (VIF = ∞): any one member can go without losing information. The rule drops the most frequent dummy, which is the coarse summary (e.g. the brand), so the finer model dummies stay.
- Otherwise the highest VIF is dropped, one at a time, until every VIF < VIF_MAX.

**Step 2: Elastic Net, with `l1_ratio` chosen by 5-fold CV on train.** Why not the alternatives:
- **Stepwise:** p-values are useless here. With ~24k rows everything is significant (section 8a of the LR notebook: p ≈ 0 for all). Stepwise is also greedy and unstable over ~600 dummies, and its post-selection p-values are biased.
- **Pure LASSO:** among correlated columns (dummies of models that share an engine size, and similar), it keeps one arbitrarily and the choice flips between samples.
- **Elastic Net:** the L2 part keeps correlated dummies together (grouping effect), and the L1 part still zeroes useless ones. Because `l1_ratio = 1.0` (= LASSO) is in the grid, CV picks LASSO if LASSO is actually better. The data decides, not us.

Columns are standardized so the penalty treats them equally. The final MLR is the **Elastic Net itself**, not an OLS refit. Many dummies have only 1–5 train rows: unpenalized OLS fits those few cars exactly and carries their noise into new cars of the same model (a first run gave OLS refit val MAE 214k vs EN 102k). The shrinkage is what makes the rare dummies usable. OLS on all columns and OLS refit are printed for comparison only.

In [2]:
Xm = {k: dummies(d, drop_first=True) for k, d in parts.items()}
m_all = Xm["train"].columns.tolist(); cols = m_all.copy(); A0 = Xm["train"]

def vif_worst(cols):
    A = A0[cols].values; A = (A - A.mean(0)) / A.std(0)
    w, V = np.linalg.eigh(A.T @ A / len(A))   # eigen-decomposition of the correlation matrix
    null = w < 1e-10 * w.max()
    if null.any():   # exact linear dependency -> VIF = inf for every column in it
        inf = [c for c, f in zip(cols, np.abs(V[:, null]).max(1) > 1e-6) if f]
        return max(inf, key=lambda c: -1 if c in NUM else A0[c].sum()), np.inf
    v = (V ** 2) @ (1 / w)   # = diag(inverse correlation matrix) = VIF
    return cols[v.argmax()], v.max()

t = time.time(); drops = []
while True:
    c, v = vif_worst(cols)
    if v < C.VIF_MAX: break
    cols.remove(c); drops.append(dict(column=c, variable=orig(c), VIF=v))
drops = pd.DataFrame(drops)
print(f"VIF: removed {len(drops)} of {len(m_all)} columns in {time.time() - t:.0f}s | max VIF left {v:.2f} (< {C.VIF_MAX})")
display(drops.assign(exact=np.isinf(drops["VIF"])).groupby(["variable", "exact"]).size().rename("n_dropped").to_frame())
drops[~np.isinf(drops["VIF"])].round(2)

VIF: removed 57 of 602 columns in 248s | max VIF left 5.45 (< 7.0)


n_dropped
variable        exact           
brand           True          50
car_type        False          3
                True           2
engine_capacity False          1
fuel_type       False          1

,column,variable,VIF
52,car_type_MPV,car_type,1215.54
53,car_type_SUV,car_type,768.46
54,car_type_Van,car_type,208.43
55,fuel_type_Diesel,fuel_type,8.84
56,engine_capacity,engine_capacity,8.14


In [3]:
sc = StandardScaler().fit(Xm["train"][cols])
S = {k: sc.transform(X[cols]) for k, X in Xm.items()}
t = time.time()
en = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 0.99, 1.0], alphas=50, cv=5, n_jobs=-1, max_iter=5000, tol=1e-3).fit(S["train"], y["train"])
mlr_sel = [c for c, b in zip(cols, en.coef_) if b != 0]
ols_all = LinearRegression().fit(Xm["train"], y["train"])
ols = LinearRegression().fit(Xm["train"][mlr_sel], y["train"])
print(f"Elastic Net ({time.time() - t:.0f}s): l1_ratio={en.l1_ratio_} ({'= LASSO' if en.l1_ratio_ == 1 else 'mix of L1+L2'}), alpha={en.alpha_:.2e}")
print(f"columns: {len(m_all)} all -> {len(cols)} after VIF -> {len(mlr_sel)} after Elastic Net")
print(f"val MAE: OLS all {vmae(ols_all.predict(Xm['val'])):,.0f} | EN {vmae(en.predict(S['val'])):,.0f} | OLS refit on selected {vmae(ols.predict(Xm['val'][mlr_sel])):,.0f}")
summary.append(dict(model="MLR", n_before=len(m_all), n_after=len(mlr_sel), val_MAE_before=vmae(ols_all.predict(Xm["val"])), val_MAE_after=vmae(en.predict(S["val"]))))

Elastic Net (203s): l1_ratio=0.1 (mix of L1+L2), alpha=1.04e-02
columns: 602 all -> 545 after VIF -> 505 after Elastic Net
val MAE: OLS all 151,743 | EN 102,147 | OLS refit on selected 214,464


## 2. DT — cost-complexity pruning
Pruning is the decision tree's own built-in (embedded) selection.
- A split survives only if its impurity reduction pays for `ccp_alpha` × the extra leaf.
- A dummy that no surviving split uses has zero effect on predictions, so it is dropped.
- Permutation or SHAP on a single tree would be unstable: one tree is high-variance, and a slightly different sample gives a different tree.

Procedure: grow the full tree, then take the pruning path's alphas (log-spaced grid of 50). Choose the **largest alpha** (smallest tree) whose val MAE is within TOL of the best.

In [4]:
Xt = {k: dummies(d, drop_first=False) for k, d in parts.items()}   # trees: keep every level
t_all = Xt["train"].columns.tolist()
used = lambda m: [t_all[j] for j in sorted(set(m.tree_.feature[m.tree_.feature >= 0]))]
t = time.time()
full_tree = DecisionTreeRegressor(random_state=C.SEED).fit(Xt["train"], y["train"])
path = full_tree.cost_complexity_pruning_path(Xt["train"], y["train"]).ccp_alphas
rows, trees = [], []
for a in np.geomspace(path[path > 0].min(), path[-2], 50):   # path[-1] prunes to the root
    m = DecisionTreeRegressor(ccp_alpha=a, random_state=C.SEED).fit(Xt["train"], y["train"])
    rows.append(dict(ccp_alpha=a, leaves=m.get_n_leaves(), n_feat=len(used(m)), val_MAE=vmae(m.predict(Xt["val"])))); trees.append(m)
dt_tune = pd.DataFrame(rows)
i = dt_tune.index[dt_tune["val_MAE"] <= dt_tune["val_MAE"].min() * (1 + TOL)][-1]
dt, dt_sel = trees[i], used(trees[i])
print(f"{time.time() - t:.0f}s | full tree: {full_tree.get_n_leaves()} leaves, {len(used(full_tree))} features used, val MAE {vmae(full_tree.predict(Xt['val'])):,.0f}")
print(f"pruned (alpha={dt_tune.loc[i, 'ccp_alpha']:.2e}): {dt.get_n_leaves()} leaves, {len(dt_sel)} features used, val MAE {dt_tune.loc[i, 'val_MAE']:,.0f}")
summary.append(dict(model="DT", n_before=len(t_all), n_after=len(dt_sel), val_MAE_before=vmae(full_tree.predict(Xt["val"])), val_MAE_after=dt_tune.loc[i, "val_MAE"]))
dt_tune.assign(chosen=dt_tune.index == i).iloc[::3].round({"val_MAE": 0})

79s | full tree: 22177 leaves, 437 features used, val MAE 96,921
pruned (alpha=5.08e-06): 1954 leaves, 320 features used, val MAE 92,638


,ccp_alpha,leaves,n_feat,val_MAE,chosen
0,1.180157e-18,22176,437,96921.0,False
3,1.248239e-17,21583,436,96921.0,False
6,1.320249e-16,21535,435,96921.0,False
9,1.396413e-15,21530,435,96921.0,False
12,1.476971e-14,21527,435,96921.0,False
15,1.562176e-13,21526,435,96921.0,False
18,1.652296e-12,21523,435,96921.0,False
21,1.747616e-11,21502,434,96921.0,False
24,1.848434e-10,21230,434,96921.0,False
27,1.955069e-09,20700,433,96926.0,False


## 3. RF — permutation importance (validation)
RF's built-in (impurity) importance is biased here. It favours continuous columns (`mileage` has thousands of split points) over 0/1 dummies, and it is computed on train, so it rewards splits the forest overfit.

Permutation importance on **validation** measures what we actually care about: how much val MAE rises when a column's link to price is broken.
- Each column is shuffled 5 times. The column is dropped if mean − 2·SD ≤ 0, i.e. its importance is not distinguishable from zero.
- A dummy whose level never varies in validation cannot be measured (shuffling a constant changes nothing). It is **kept**, because "unmeasured" is not the same as "useless"; it may matter on test.

The forest is then refit on the kept columns with the same hyperparameters.

In [5]:
rf_kw = dict(n_estimators=200, max_features=1 / 3, n_jobs=-1, random_state=C.SEED)
t = time.time()
rf = RandomForestRegressor(**rf_kw).fit(Xt["train"], y["train"])
V = Xt["val"].copy(); base = vmae(rf.predict(V))
measurable = [c for c in t_all if V[c].nunique() > 1]
rng = np.random.default_rng(C.SEED); imp = {}
for c in measurable:
    keep_vals, s = V[c].values.copy(), []
    for _ in range(5):
        V[c] = rng.permutation(keep_vals); s.append(vmae(rf.predict(V)) - base)
    V[c] = keep_vals; imp[c] = (np.mean(s), np.std(s))
imp = pd.DataFrame(imp, index=["MAE_increase", "SD"]).T.sort_values("MAE_increase", ascending=False)
drop_rf = imp.index[imp["MAE_increase"] - 2 * imp["SD"] <= 0]
rf_sel = [c for c in t_all if c not in set(drop_rf)]
rf2 = RandomForestRegressor(**rf_kw).fit(Xt["train"][rf_sel], y["train"])
print(f"{time.time() - t:.0f}s | measurable {len(measurable)} of {len(t_all)} | dropped {len(drop_rf)} | kept {len(rf_sel)} ({len(t_all) - len(measurable)} unmeasurable kept)")
print(f"val MAE: all {base:,.0f} -> selected {vmae(rf2.predict(Xt['val'][rf_sel])):,.0f}")
summary.append(dict(model="RF", n_before=len(t_all), n_after=len(rf_sel), val_MAE_before=base, val_MAE_after=vmae(rf2.predict(Xt["val"][rf_sel]))))
imp.head(15).round(0)

223s | measurable 351 of 608 | dropped 90 | kept 518 (257 unmeasurable kept)
val MAE: all 81,616 -> selected 82,528


,MAE_increase,SD
engine_capacity,188417.0,4779.0
car_age,176992.0,3079.0
mileage,46562.0,1626.0
brand_MERCEDES-BENZ,32289.0,731.0
car_type_Pickup,30574.0,2323.0
fuel_type_Benzine,28192.0,788.0
car_type_Van,22447.0,558.0
fuel_type_Hybrid,11000.0,683.0
brand_BMW,8085.0,307.0
fuel_type_Diesel,6779.0,443.0


## 4. XGB — early stopping + SHAP
- **Early stopping (val)** fixes the number of trees at the point where val loss stops improving. Importance is then read from a model that is not overfit; an overfit booster spreads importance onto noise columns.
- **SHAP** (TreeSHAP, via xgboost's built-in `pred_contribs`) is used instead of gain/weight importance. Gain is inconsistent (a feature can matter more yet score lower) and favours columns with many split points. SHAP attributions are consistent and add up to each prediction.
- Ranking: mean |SHAP| on **train**. Every level appears in train, and SHAP describes the fitted model, not generalization.
- Cutoff: keep the top-k columns that reach 90 / 95 / 98 / 99 / 99.9 / 100% of total |SHAP|. Refit each with early stopping, then pick the **smallest k** whose val MAE is within TOL of the best.

In [6]:
xkw = dict(n_estimators=5000, learning_rate=0.05, max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
           early_stopping_rounds=100, tree_method="hist", random_state=C.SEED, n_jobs=-1)
# ponytail: same val set for early stopping + cutoff choice, test is the honest check
fit_xgb = lambda cols: xgb.XGBRegressor(**xkw).fit(Xt["train"][cols], y["train"], eval_set=[(Xt["val"][cols], y["val"])], verbose=False)
t = time.time()
xg = fit_xgb(t_all)
shap = xg.get_booster().predict(xgb.DMatrix(Xt["train"]), pred_contribs=True, iteration_range=(0, xg.best_iteration + 1))[:, :-1]
ms = pd.Series(np.abs(shap).mean(0), index=t_all).sort_values(ascending=False)
share = ms.cumsum() / ms.sum()
rows, boosters = [], []
for q in [0.9, 0.95, 0.98, 0.99, 0.999, 1.0]:
    k = int((ms > 0).sum()) if q == 1 else int((share < q).sum()) + 1
    m = fit_xgb(ms.index[:k].tolist())
    rows.append(dict(shap_share=q, n_feat=k, n_trees=m.best_iteration + 1, val_MAE=vmae(m.predict(Xt["val"][ms.index[:k]])))); boosters.append(m)
xg_tune = pd.DataFrame(rows)
j = xg_tune.index[xg_tune["val_MAE"] <= xg_tune["val_MAE"].min() * (1 + TOL)][0]
xg2, xg_sel = boosters[j], ms.index[:xg_tune.loc[j, "n_feat"]].tolist()
print(f"{time.time() - t:.0f}s | all {len(t_all)} cols: {xg.best_iteration + 1} trees, val MAE {vmae(xg.predict(Xt['val'])):,.0f} | SHAP = 0 for {(ms == 0).sum()} cols")
print(f"chosen: top {len(xg_sel)} cols ({xg_tune.loc[j, 'shap_share']:.1%} of |SHAP|), val MAE {xg_tune.loc[j, 'val_MAE']:,.0f}")
summary.append(dict(model="XGB", n_before=len(t_all), n_after=len(xg_sel), val_MAE_before=vmae(xg.predict(Xt["val"])), val_MAE_after=xg_tune.loc[j, "val_MAE"]))
display(xg_tune.assign(chosen=xg_tune.index == j).round({"val_MAE": 0}))
ms.head(15).round(4).rename("mean |SHAP|").to_frame()

106s | all 608 cols: 1329 trees, val MAE 85,151 | SHAP = 0 for 289 cols
chosen: top 171 cols (99.0% of |SHAP|), val MAE 84,071


,shap_share,n_feat,n_trees,val_MAE,chosen
0,0.900,36,1348,96056.0,False
1,0.950,74,1290,92315.0,False
2,0.980,132,1294,88382.0,False
3,0.990,171,2405,84071.0,True
4,0.999,262,2545,83401.0,False
5,1.000,319,1701,84010.0,False


,mean |SHAP|
car_age,0.3006
engine_capacity,0.2176
car_type_Pickup,0.0728
mileage,0.0616
fuel_type_Benzine,0.0513
brand_MERCEDES-BENZ,0.0469
car_type_Van,0.0316
brand_MAZDA,0.0264
brand_BMW,0.0240
gear_type_Automatic,0.0182


## 5. Summary
Columns kept per original variable (kept / total dummies). MLR totals are one lower per categorical because of `drop_first`.

In [7]:
s = pd.DataFrame(summary).set_index("model")
s["val_MAE_change"] = (s["val_MAE_after"] / s["val_MAE_before"] - 1).map("{:+.2%}".format)
display(s.round(0))
pd.DataFrame({"MLR": kept_by_var(mlr_sel, m_all), "DT": kept_by_var(dt_sel, t_all), "RF": kept_by_var(rf_sel, t_all), "XGB": kept_by_var(xg_sel, t_all)})

,n_before,n_after,val_MAE_before,val_MAE_after,val_MAE_change
model,,,,,
MLR,602,505,151743.0,102147.0,-32.68%
DT,608,320,96921.0,92638.0,-4.42%
RF,608,518,81616.0,82528.0,+1.12%
XGB,608,171,85151.0,84071.0,-1.27%


,MLR,DT,RF,XGB
brand,1/51,33/52,47/52,29/52
car_age,1/1,1/1,1/1,1/1
car_type,0/6,7/7,7/7,7/7
color,15/20,19/21,7/21,14/21
engine_capacity,0/1,1/1,1/1,1/1
fuel_type,1/2,3/3,3/3,3/3
gear_type,1/1,2/2,2/2,2/2
mileage,1/1,1/1,1/1,1/1
model,485/519,253/520,449/520,113/520


## 6. Test — final models, scored once

In [8]:
yt = test[C.TARGET]
pd.DataFrame([score("MLR (Elastic Net on VIF columns)", yt, np.exp(en.predict(S["test"]))),
              score("DT (pruned)", yt, np.exp(dt.predict(Xt["test"]))),
              score("RF (permutation-selected)", yt, np.exp(rf2.predict(Xt["test"][rf_sel]))),
              score("XGB (SHAP-selected)", yt, np.exp(xg2.predict(Xt["test"][xg_sel])))]).set_index("model").round(4)

,RMSE,MAE,MAPE,R2
model,,,,
MLR (Elastic Net on VIF columns),355815.9050,108339.8135,0.1519,0.7896
DT (pruned),325649.7892,97422.0705,0.1388,0.8238
RF (permutation-selected),275419.6940,85568.5159,0.1215,0.8740
XGB (SHAP-selected),355138.7969,86394.3594,0.1166,0.7904
